## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

In [3]:
# ✅ Agents developed under `agentic/agents`:
#   - classifier.py  : Deterministic ticket classification (category + urgency)
#   - resolver.py    : Knowledge-base retrieval + tool-assisted resolution
#   - escalation.py  : Builds escalation packages for unresolvable/critical issues
#   - memory_agent.py: Long-term memory - retrieves/stores customer interaction history
#
# ✅ Tools developed under `agentic/tools`:
#   - account_tools.py : lookup_user_record, get_subscription_details
#   - ticket_tools.py  : update_ticket_status, log_ticket_message, get_ticket_history
#   - knowledge_tools.py: search_knowledge_base (semantic retrieval from articles)
#
# ✅ Workflow orchestration in `agentic/workflow.py`:
#   - Supervisor pattern using LangGraph StateGraph
#   - Entry: memory → supervisor → classifier → supervisor → resolver/escalation → END
#   - MemorySaver checkpointer for session & thread-level state persistence
#   - Conditional routing based on classification, urgency, and resolution confidence

# Verify agent modules are importable
from agentic.agents.classifier import classify_ticket
from agentic.agents.resolver import resolve_ticket
from agentic.agents.escalation import build_escalation
from agentic.agents.memory_agent import resolve_customer_context
from agentic.tools.account_tools import lookup_user_record
from agentic.tools.ticket_tools import update_ticket_status, log_ticket_message, get_ticket_history
from agentic.tools.knowledge_tools import search_knowledge_base

print("✅ All agents and tools imported successfully")
print(f"   Agents: classifier, resolver, escalation, memory_agent")
print(f"   Tools: lookup_user_record, update_ticket_status, log_ticket_message, get_ticket_history, search_knowledge_base")

✅ All agents and tools imported successfully
   Agents: classifier, resolver, escalation, memory_agent
   Tools: lookup_user_record, update_ticket_status, log_ticket_message, get_ticket_history, search_knowledge_base


In [4]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

# Reload to ensure latest code is used (useful during development)
import importlib, agentic.workflow_logger, agentic.workflow
importlib.reload(agentic.workflow_logger)
importlib.reload(agentic.workflow)

from agentic.workflow import orchestrator

## Architecture Overview

The multi-agent system uses a **Supervisor pattern** orchestrated with LangGraph:

```
User Message → [Memory Agent] → [Supervisor] → [Classifier] → [Supervisor] → [Resolver/Escalation] → Response
```

| Agent | Role | Responsibility |
|-------|------|----------------|
| **Memory** | Context retrieval | Matches users, retrieves past interactions, stores preferences |
| **Classifier** | Routing | Categorizes tickets (billing/technical/account/etc.) and assigns urgency |
| **Resolver** | Resolution | Searches knowledge base, invokes tools, generates responses |
| **Escalation** | Fallback | Creates escalation packages for critical/unresolvable issues |
| **Supervisor** | Orchestrator | Routes between agents based on state (classification, confidence, urgency) |

**State persistence**: `MemorySaver` checkpointer enables session inspection via `thread_id`.

In [5]:
from collections import Counter

from agentic.workflow_logger import clear_logs, get_logs, search_logs
from langchain_core.messages import HumanMessage

clear_logs()

sample_tickets = [
    {
        "thread_id": "demo-ticket-1",
        "label": "Technical Issue - Login Problem",
        "expected": "resolution",
        "message": "I can't log in to my CultPass account. The app keeps crashing when I enter my password.",
    },
    {
        "thread_id": "demo-ticket-2",
        "label": "Billing Issue - Subscription Question",
        "expected": "resolution",
        "message": "I was charged twice for my monthly subscription. Can you help me understand my billing?",
    },
    {
        "thread_id": "demo-ticket-3",
        "label": "Account Security - Hacked Account",
        "expected": "escalation",
        "message": "My account has been hacked! Someone changed my email and password. I think my credit card info was stolen.",
    },
    {
        "thread_id": "demo-ticket-4",
        "label": "Reservation Issue - Event Booking Help",
        "expected": "resolution",
        "message": "I need help with my reservation. How do I cancel a booking and get a refund for an upcoming event?",
    },
    {
        "thread_id": "demo-ticket-5",
        "label": "General / Edge Case - Vague Question",
        "expected": "edge_case",
        "message": "What is CultPass and how does your energy optimization feature work?",
    },
]

runtime_ticket_summaries = []

for index, ticket in enumerate(sample_tickets, start=1):
    print("=" * 70)
    print(f"TICKET {index}: {ticket['label']}")
    print(f"Expected outcome: {ticket['expected']}")
    print("=" * 70)

    result = orchestrator.invoke(
        {"messages": [HumanMessage(content=ticket["message"])]},
        config={"configurable": {"thread_id": ticket["thread_id"]}},
    )

    thread_logs = get_logs(thread_id=ticket["thread_id"])
    log_counts = Counter(event["event_type"] for event in thread_logs)
    resolution_event = next((event for event in reversed(thread_logs) if event["event_type"] == "resolution_attempt"), None)
    escalation_event = next((event for event in reversed(thread_logs) if event["event_type"] == "escalation"), None)
    routing_targets = [event["route"] for event in thread_logs if event.get("route")]
    tool_names = [event["tool_name"] for event in thread_logs if event.get("tool_name")]
    final_action = escalation_event["final_action"] if escalation_event else (resolution_event["final_action"] if resolution_event else "unknown")

    summary = {
        "thread_id": ticket["thread_id"],
        "label": ticket["label"],
        "classification": result.get("classification"),
        "urgency": result.get("urgency"),
        "confidence": result.get("resolution_confidence"),
        "final_action": final_action,
        "ticket_id": thread_logs[0]["ticket_id"] if thread_logs else "",
        "event_count": len(thread_logs),
        "event_types": dict(log_counts),
        "routes": routing_targets,
        "tools": tool_names,
        "response_preview": result["messages"][-1].content[:180],
    }
    runtime_ticket_summaries.append(summary)

    print(f"Runtime thread id  : {summary['thread_id']}")
    print(f"Runtime ticket id  : {summary['ticket_id']}")
    print(f"Classification     : {summary['classification']}")
    print(f"Urgency            : {summary['urgency']}")
    print(f"Confidence         : {summary['confidence']}")
    print(f"Final action       : {summary['final_action']}")
    print(f"Routes observed    : {summary['routes']}")
    print(f"Tools observed     : {summary['tools']}")
    print(f"Structured events  : {summary['event_types']}")
    print(f"Response preview   : {summary['response_preview']}...")
    print()

resolution_count = sum(1 for item in runtime_ticket_summaries if item["final_action"] == "resolved")
escalation_count = sum(1 for item in runtime_ticket_summaries if item["final_action"] == "escalated_to_human")

print("=" * 70)
print("RUNTIME OUTCOME SUMMARY")
print("=" * 70)
print(f"Tickets processed  : {len(runtime_ticket_summaries)}")
print(f"Resolved tickets   : {resolution_count}")
print(f"Escalated tickets  : {escalation_count}")
print(f"Threads with logs  : {[item['thread_id'] for item in runtime_ticket_summaries]}")

TICKET 1: Technical Issue - Login Problem
Expected outcome: resolution
Runtime thread id  : demo-ticket-1
Runtime ticket id  : ticket-demo-ticket-1
Classification     : technical
Urgency            : high
Confidence         : MEDIUM
Final action       : resolved
Routes observed    : ['classifier', 'resolver', 'END']
Tools observed     : ['search_knowledge_base']
Structured events  : {'memory_retrieval': 1, 'routing_decision': 3, 'classification': 1, 'tool_usage': 1, 'resolution_attempt': 1}
Response preview   : I found relevant guidance in What's Included in a CultPass Subscription, How to Handle Blocked Accounts.

Each user is entitled to 4 cultural experiences per month, which may inclu...

TICKET 2: Billing Issue - Subscription Question
Expected outcome: resolution
Runtime thread id  : demo-ticket-2
Runtime ticket id  : ticket-demo-ticket-2
Classification     : billing
Urgency            : medium
Confidence         : MEDIUM
Final action       : resolved
Routes observed    : ['classi

## Structured Workflow Logs — Verification

The workflow logger persists **structured, searchable** log events to `logs/workflow_events.jsonl`.  
Each record includes: `event_type`, `ticket_id`, `agent`, `decision`, `route`, `tool_name`, `tool_result_summary`, `confidence`, `final_action`, `timestamp`, and more.

Below we verify the logs were created for all demo tickets above.

In [6]:
# ============================================================
# VERIFY STRUCTURED LOGS
# ============================================================
import json

all_logs = get_logs()
print(f"Total structured log events recorded: {len(all_logs)}\n")

from collections import Counter

event_types = Counter(event["event_type"] for event in all_logs)
print("Event Type Distribution:")
for event_type, count in event_types.most_common():
    print(f"  {event_type:25s} : {count}")

print(f"\n{'=' * 70}")
print("PER-THREAD RUNTIME VERIFICATION")
print(f"{'=' * 70}")
for item in runtime_ticket_summaries:
    thread_logs = get_logs(thread_id=item["thread_id"])
    unique_thread_ids = sorted({event["thread_id"] for event in thread_logs})
    unique_ticket_ids = sorted({event["ticket_id"] for event in thread_logs})
    print(f"{item['thread_id']}: {len(thread_logs)} event(s)")
    print(f"  Thread ids seen : {unique_thread_ids}")
    print(f"  Ticket ids seen : {unique_ticket_ids}")
    print(f"  Event types     : {sorted({event['event_type'] for event in thread_logs})}")

print(f"\n{'=' * 70}")
print("SAMPLE STRUCTURED LOG RECORDS (first 6 runtime events)")
print(f"{'=' * 70}\n")
for index, event in enumerate(all_logs[:6], start=1):
    print(f"[{index}] {json.dumps(event, indent=2)}\n")

Total structured log events recorded: 34

Event Type Distribution:
  routing_decision          : 15
  memory_retrieval          : 5
  classification            : 5
  tool_usage                : 4
  resolution_attempt        : 4
  escalation                : 1

PER-THREAD RUNTIME VERIFICATION
demo-ticket-1: 7 event(s)
  Thread ids seen : ['demo-ticket-1']
  Ticket ids seen : ['ticket-demo-ticket-1']
  Event types     : ['classification', 'memory_retrieval', 'resolution_attempt', 'routing_decision', 'tool_usage']
demo-ticket-2: 7 event(s)
  Thread ids seen : ['demo-ticket-2']
  Ticket ids seen : ['ticket-demo-ticket-2']
  Event types     : ['classification', 'memory_retrieval', 'resolution_attempt', 'routing_decision', 'tool_usage']
demo-ticket-3: 6 event(s)
  Thread ids seen : ['demo-ticket-3']
  Ticket ids seen : ['ticket-demo-ticket-3']
  Event types     : ['classification', 'escalation', 'memory_retrieval', 'routing_decision']
demo-ticket-4: 7 event(s)
  Thread ids seen : ['demo-tick

In [7]:
# ============================================================
# VERIFY: Logs are searchable/filterable by structured fields
# ============================================================

print("=" * 70)
print("FILTER TEST 1: Classification events for technical tickets")
print("=" * 70)
classification_logs = search_logs(event_type="classification", category="technical")
for log in classification_logs:
    print(f"  Thread: {log['thread_id']:14s} | Category: {log['category']:12s} | Urgency: {log['urgency']:8s} | Decision: {log['decision']}")

print(f"\n{'=' * 70}")
print("FILTER TEST 2: Escalation events")
print(f"{'=' * 70}")
escalation_logs = get_logs(event_type="escalation")
for log in escalation_logs:
    print(f"  Thread: {log['thread_id']:14s} | Decision: {log['decision']} | Final Action: {log['final_action']} | Urgency: {log['urgency']}")

print(f"\n{'=' * 70}")
print("FILTER TEST 3: Tool usage events")
print(f"{'=' * 70}")
tool_logs = get_logs(event_type="tool_usage")
for log in tool_logs:
    print(f"  Thread: {log['thread_id']:14s} | Tool: {log['tool_name']:30s} | Result Preview: {log['tool_result_summary'][:70]}")

print(f"\n{'=' * 70}")
print("FILTER TEST 4: Routing decisions that reached escalation")
print(f"{'=' * 70}")
routing_logs = search_logs(event_type="routing_decision", route="escalation")
for log in routing_logs:
    print(f"  Thread: {log['thread_id']:14s} | Route: {log['route']:10s} | Decision: {log['decision']:30s} | Confidence: {log.get('confidence', '-')}")

print(f"\n{'=' * 70}")
print("FILTER TEST 5: Resolution attempts that resolved successfully")
print(f"{'=' * 70}")
resolution_logs = search_logs(event_type="resolution_attempt", final_action="resolved")
for log in resolution_logs:
    print(f"  Thread: {log['thread_id']:14s} | Confidence: {log['confidence']:6s} | Final Action: {log['final_action']:10s} | Category: {log['category']}")

FILTER TEST 1: Classification events for technical tickets
  Thread: demo-ticket-1  | Category: technical    | Urgency: high     | Decision: classified_as_technical

FILTER TEST 2: Escalation events
  Thread: demo-ticket-3  | Decision: escalated_P1 | Final Action: escalated_to_human | Urgency: critical

FILTER TEST 3: Tool usage events
  Thread: demo-ticket-1  | Tool: search_knowledge_base          | Result Preview: Found 3 relevant article(s):

--- Article 1 (Relevance: 0.294) ---
Tit
  Thread: demo-ticket-2  | Tool: search_knowledge_base          | Result Preview: Found 3 relevant article(s):

--- Article 1 (Relevance: 0.262) ---
Tit
  Thread: demo-ticket-4  | Tool: search_knowledge_base          | Result Preview: Found 3 relevant article(s):

--- Article 1 (Relevance: 0.544) ---
Tit
  Thread: demo-ticket-5  | Tool: search_knowledge_base          | Result Preview: Found 3 relevant article(s):

--- Article 1 (Relevance: 0.262) ---
Tit

FILTER TEST 4: Routing decisions that reached esc

In [8]:
# ============================================================
# VERIFY: Structured logs are persisted to disk as JSONL
# ============================================================
import os

log_path = os.path.join("logs", "workflow_events.jsonl")
assert os.path.exists(log_path), f"Log file not found at {log_path}"

with open(log_path, "r", encoding="utf-8") as handle:
    lines = [line.strip() for line in handle if line.strip()]

print(f"Log file exists        : {log_path}")
print(f"Persisted log records  : {len(lines)}")

required_fields = {
    "timestamp",
    "event_type",
    "agent",
    "ticket_id",
    "thread_id",
    "decision",
    "route",
    "tool_name",
    "tool_result_summary",
    "confidence",
    "final_action",
}
parsed_records = []
for index, line in enumerate(lines, start=1):
    record = json.loads(line)
    parsed_records.append(record)
    missing = required_fields - set(record.keys())
    assert not missing, f"Record {index} missing fields: {missing}"
    assert record["thread_id"], f"Record {index} is missing runtime thread_id"
    assert record["ticket_id"], f"Record {index} is missing runtime ticket_id"

state_history_checks = {}
for item in runtime_ticket_summaries:
    history = list(orchestrator.get_state_history({"configurable": {"thread_id": item["thread_id"]}}))
    latest_values = history[0].values
    state_history_checks[item["thread_id"]] = {
        "history_entries": len(history),
        "workflow_events_in_state": len(latest_values.get("workflow_events", [])),
    }

resolved_threads = sorted({event["thread_id"] for event in search_logs(event_type="resolution_attempt", final_action="resolved")})
escalated_threads = sorted({event["thread_id"] for event in search_logs(event_type="escalation", final_action="escalated_to_human")})

print(f"All records valid JSON : PASS")
print(f"Required fields check  : PASS")
print(f"Runtime ids present    : PASS")
print()
print("ASSIGNMENT REQUIREMENTS - RUNTIME EVIDENCE")
print("=" * 70)
print(f"Complete flow shown            : {len(runtime_ticket_summaries)} tickets processed end-to-end")
print(f"Successful resolutions         : {len(resolved_threads)} thread(s) -> {resolved_threads}")
print(f"Escalation scenarios           : {len(escalated_threads)} thread(s) -> {escalated_threads}")
print(f"Classification events          : {len(get_logs(event_type='classification'))}")
print(f"Routing decision events        : {len(get_logs(event_type='routing_decision'))}")
print(f"Tool usage events              : {len(get_logs(event_type='tool_usage'))}")
print(f"Resolution attempt events      : {len(get_logs(event_type='resolution_attempt'))}")
print(f"Escalation events              : {len(get_logs(event_type='escalation'))}")
print(f"State history contains events  : {state_history_checks}")
print(f"Searchable JSONL persistence   : {log_path}")

Log file exists        : logs\workflow_events.jsonl
Persisted log records  : 34
All records valid JSON : PASS
Required fields check  : PASS
Runtime ids present    : PASS

ASSIGNMENT REQUIREMENTS - RUNTIME EVIDENCE
Complete flow shown            : 5 tickets processed end-to-end
Successful resolutions         : 4 thread(s) -> ['demo-ticket-1', 'demo-ticket-2', 'demo-ticket-4', 'demo-ticket-5']
Escalation scenarios           : 1 thread(s) -> ['demo-ticket-3']
Classification events          : 5
Routing decision events        : 15
Tool usage events              : 4
Resolution attempt events      : 4
Escalation events              : 1
State history contains events  : {'demo-ticket-1': {'history_entries': 8, 'workflow_events_in_state': 7}, 'demo-ticket-2': {'history_entries': 8, 'workflow_events_in_state': 7}, 'demo-ticket-3': {'history_entries': 8, 'workflow_events_in_state': 6}, 'demo-ticket-4': {'history_entries': 8, 'workflow_events_in_state': 7}, 'demo-ticket-5': {'history_entries': 8, '

In [9]:
# ============================================================
# FULL FORMATTED LOG OUTPUT (human-readable)
# ============================================================
from agentic.workflow_logger import get_logs_summary

print(get_logs_summary())

STRUCTURED WORKFLOW LOG — 34 event(s)

[1] 2026-05-24T16:38:19.873330+00:00
    Event Type : memory_retrieval
    Agent      : memory
    Ticket ID  : ticket-demo-ticket-1
    Thread ID  : demo-ticket-1
    Decision   : no_context_found

[2] 2026-05-24T16:38:19.876976+00:00
    Event Type : routing_decision
    Agent      : supervisor
    Ticket ID  : ticket-demo-ticket-1
    Thread ID  : demo-ticket-1
    Decision   : needs_classification
    Route      : classifier

[3] 2026-05-24T16:38:19.879253+00:00
    Event Type : classification
    Agent      : classifier
    Ticket ID  : ticket-demo-ticket-1
    Thread ID  : demo-ticket-1
    Decision   : classified_as_technical
    Category   : technical
    Urgency    : high

[4] 2026-05-24T16:38:19.881230+00:00
    Event Type : routing_decision
    Agent      : supervisor
    Ticket ID  : ticket-demo-ticket-1
    Thread ID  : demo-ticket-1
    Decision   : attempt_resolution
    Route      : resolver
    Category   : technical
    Urgency  

In [13]:
chat_interface(orchestrator, "1")

User: hi
Assistant: I understand your concern, and I want to make sure you get the best possible help.

I've escalated your case to our specialized support team with P4 priority.

Summary: General issue: hi

Your case has been escalated with P4 priority. A human support specialist will review your general issue and follow up shortly.
User: ok
Assistant: I understand your concern, and I want to make sure you get the best possible help.

I've escalated your case to our specialized support team with P4 priority.

Summary: General issue: ok

Your case has been escalated with P4 priority. A human support specialist will review your general issue and follow up shortly.
User: Thanks
Assistant: I understand your concern, and I want to make sure you get the best possible help.

I've escalated your case to our specialized support team with P4 priority.

Summary: General issue: Thanks

Your case has been escalated with P4 priority. A human support specialist will review your general issue and fol

In [12]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='[Memory] No prior customer history found.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content='[Classifier] Category: general, Urgency: low', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content='I could not find a reliable knowledge base article for this request. I recommend escalating it to a human support specialist so they can investigate further.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content="I understand your concern, and I want to make sure you get the best possible help.\n\nI've escalated your case to our specialized support team with P4 priority.\n\nSummary: General issue: hi\n\nYour case has been escalated with P4 priority. A human support specialist will review your general issue and follow up shortly.", additional_kwargs={}, r